# Prepare Human Comments Reference

This notebook builds the cleaned human-comment reference set from `uicrit_notna_deduped.parquet`.

It produces one main output:
- `human_critiques_final.parquet`: one row per canonical human comment per screen, with linked task ids

These outputs are intended for later LLM-vs-human comment matching within the same screen.

### **Notes from expert-critique inspection across tasks within the same screen**

A manual inspection of multiple screens with several tasks each suggests that expert critiques are often **screen-general rather than strictly task-specific**. Across tasks for the same screen, many comments recur with only wording changes, especially around **font size/readability, contrast, spacing, alignment, clutter, hierarchy/prominence, unclear icons, and missing navigation elements**.

The inspection also suggests that expert comments are **not always unique at the issue level**. Many critiques appear to be **rephrased, overlapping, or partially redundant**, either within the same task or across different tasks of the same screen. In several cases, the task wording changes, but the underlying critique remains essentially about the same screen-level UI problem.

Based on this exploratory review, relying on only **one task per screen** may miss part of the expert critique space. For the next stage, it is reasonable to treat **pooled expert critiques at the screen level** as an important reference for LLM comparison, while keeping task-level comparison as a secondary or sensitivity analysis.

In [ ]:
from pathlib import Path
import re

import numpy as np
import json
import pandas as pd

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", 50)

DATA_DIR = Path("./cleaned_dataset")
INPUT_PATH = DATA_DIR / "uicrit_notna_deduped.parquet"
OUTPUT_DIR = DATA_DIR / "human_critiques"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DROP_LLM = True
DROP_MISSING_OBSERVED = True
DROP_MISSING_EXPECTED = False
DROP_MISSING_FIX = False

DEDUP_WITHIN_TASK = True
DEDUP_WITHIN_SCREEN = True

TASK_DEDUP_KEY = ["screen_task_id", "observed_issue_norm"]
SCREEN_DEDUP_KEY = ["screen_id", "observed_issue_norm"]

INPUT_PATH

WindowsPath('cleaned_dataset/uicrit_notna_deduped.parquet')

## 1. Removing Missing and Exact Duplicate Comments

### 1.1 Load Parsed Comments

The input table is currently at screen-task level, with `parsed_comments` stored as a list of dictionaries.

In [2]:
uicrit_df = pd.read_parquet(INPUT_PATH)

print("Rows (screen-task level):", len(uicrit_df))
print("Unique screens:", uicrit_df["screen_id"].nunique())
print("Unique screen-task ids:", uicrit_df["screen_task_id"].nunique())
print("Rows with parsed comments:", uicrit_df["parsed_comments"].notna().sum())

uicrit_df[["screen_id", "app_category", "screen_task_id", "task", "parsed_comments"]].head(2)

Rows (screen-task level): 2957
Unique screens: 1000
Unique screen-task ids: 2957
Rows with parsed comments: 2957


,screen_id,app_category,screen_task_id,task,parsed_comments
0,15,Health & Fitness,15_T01,Plan and Start Full Body Workouts,"[{'bounding_box': {'x_max': 0.40545597, 'x_min': 0.1559446, 'y_max': 0.09649163, 'y_min': 0.05137866}, 'comment_label': 'Comment 1', 'comment_source': 'Huma..."
1,15,Health & Fitness,15_T02,Tap on the Plus icon or explore the Workout Plans.,"[{'bounding_box': {'x_max': 0.10555714, 'x_min': 0.02777819, 'y_max': 0.10937545, 'y_min': 0.04375018}, 'comment_label': 'Comment 1', 'comment_source': 'Hum..."


### 1.2 Flatten To Comment-Level Long Format

Each parsed comment becomes one row while preserving screen-level and task-level provenance.

In [3]:
base_cols = [
    "screen_id",
    "screen_task_id",
    "task",
    "parsed_comments",
]

comments_long = (
    uicrit_df[base_cols]
    .explode("parsed_comments", ignore_index=True)
    .rename(columns={"parsed_comments": "parsed_comment"})
)

comments_long = comments_long[comments_long["parsed_comment"].notna()].copy()

# Preserve the original nested bounding_box dict before flattening the rest.
comments_long["bounding_box"] = comments_long["parsed_comment"].apply(
    lambda x: x.get("bounding_box") if isinstance(x, dict) else None
)

comment_fields = comments_long["parsed_comment"].apply(lambda x: x if isinstance(x, dict) else {}).apply(
    lambda d: {k: v for k, v in d.items() if k != "bounding_box"}
)
comment_fields = pd.json_normalize(comment_fields)

comments_long = pd.concat(
    [comments_long.drop(columns=["parsed_comment"]).reset_index(drop=True), comment_fields.reset_index(drop=True)],
    axis=1,
)

comments_long.insert(0, "comment_row_id", np.arange(1, len(comments_long) + 1))

print("Comment-level rows:", len(comments_long))
comments_long.head(3)


Comment-level rows: 11365


,comment_row_id,screen_id,screen_task_id,task,bounding_box,comment_label,comment_source,expected_standard,missing_parts,observed_issue,raw_text,suggested_fix
0,1,15,15_T01,Plan and Start Full Body Workouts,"{'x_max': 0.40545597, 'x_min': 0.1559446, 'y_max': 0.09649163, 'y_min': 0.05137866}",Comment 1,Human,the text’s visual treatment and formatting should make it easy to understand.,[],the text (Workouts) is small even when it is heading.,"Comment 1 The expected standard is that the text’s visual treatment and formatting should make it easy to understand. In the current design, the text (Worko...",increase font size and weight to make it look like a heading
1,2,15,15_T01,Plan and Start Full Body Workouts,"{'x_max': 0.1559446, 'x_min': 0.02896114, 'y_max': 0.18045189, 'y_min': 0.15037657}",Comment 2,Human,the text and background colors used in the design should be complementary and easy to read.,[],text (Plans) is in light red color on white background which is not making a good contrast.,"Comment 2 The expected standard is that the text and background colors used in the design should be complementary and easy to read. In the current design, t...",change colors to be more complementary to each other (change texts to dark colors) to make it a good contrast and easier to read.
2,3,15,15_T01,Plan and Start Full Body Workouts,"{'x_max': 0.1002501, 'x_min': 0.03787226, 'y_max': 0.09147908, 'y_min': 0.05513808}",Comment 3,Human,the design should make the most important information visually dominant.,[],the back button size is small.,"Comment 3 The expected standard is that the design should make the most important information visually dominant. In the current design, the back button size...",increase the back button size to make it visually prominent for the users.


### 1.3 Initial Profiling

Before cleaning, inspect source mix, missingness, and volume.

In [4]:
profile_summary = pd.Series({
    "total_comments": len(comments_long),
    "unique_screens": comments_long["screen_id"].nunique(),
    "unique_screen_tasks": comments_long["screen_task_id"].nunique(),
    "human_comments": (comments_long["comment_source"] == "Human").sum(),
    "llm_comments": (comments_long["comment_source"] == "LLM").sum(),
    "missing_expected_standard": comments_long["expected_standard"].isna().sum(),
    "missing_observed_issue": comments_long["observed_issue"].isna().sum(),
    "missing_suggested_fix": comments_long["suggested_fix"].isna().sum(),
})

profile_summary.to_frame("count")

,count
total_comments,11365
unique_screens,1000
unique_screen_tasks,2954
human_comments,8277
llm_comments,3088
missing_expected_standard,12
missing_observed_issue,7
missing_suggested_fix,17


In [5]:
comments_per_task = comments_long.groupby("screen_task_id").size().rename("num_comments")
comments_per_screen = comments_long.groupby("screen_id").size().rename("num_comments")

print("Comments per task")
print(comments_per_task.describe())
print()
print("Comments per screen")
print(comments_per_screen.describe())

Comments per task
count    2954.000000
mean        3.847326
std         1.924774
min         1.000000
25%         2.000000
50%         4.000000
75%         5.000000
max        13.000000
Name: num_comments, dtype: float64

Comments per screen
count    1000.000000
mean       11.365000
std         3.654193
min         3.000000
25%         9.000000
50%        11.000000
75%        14.000000
max        27.000000
Name: num_comments, dtype: float64


In [6]:
# Check which screen-task rows from uicrit_df do not appear in comments_long

all_task_ids = set(uicrit_df["screen_task_id"].dropna().astype(str))
exploded_task_ids = set(comments_long["screen_task_id"].dropna().astype(str))

missing_task_ids = sorted(all_task_ids - exploded_task_ids)

print("Number of screen_task_id values missing from comments_long:", len(missing_task_ids))
print("Missing screen_task_id values:")
print(missing_task_ids)

missing_rows = (
    uicrit_df[uicrit_df["screen_task_id"].astype(str).isin(missing_task_ids)]
    .loc[:, ["screen_id", "screen_task_id", "task", "parsed_comments"]]
    .sort_values(["screen_id", "screen_task_id"])
    .reset_index(drop=True)
)

display(missing_rows)

Number of screen_task_id values missing from comments_long: 3
Missing screen_task_id values:
['26804_T03', '27590_T02', '58424_T01']


,screen_id,screen_task_id,task,parsed_comments
0,26804,26804_T03,choose the Pokemon from the list,[]
1,27590,27590_T02,Select a template,[]
2,58424,58424_T01,Login using Google or Facebook,[]


### 1.4 Normalize Text Fields

Normalization supports exact deduplication and later matching. Keep both original and normalized forms.

In [7]:
TEXT_FIELDS = ["expected_standard", "observed_issue", "suggested_fix", "raw_text"]

def normalize_text(value):
    if value is None:
        return None
    if isinstance(value, float) and pd.isna(value):
        return None

    text = str(value).strip()
    if not text:
        return None

    text = text.lower()
    text = text.replace("\u2019", "'")
    text = text.replace("\u2018", "'")
    text = text.replace("\u201c", '"')
    text = text.replace("\u201d", '"')
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"^[^\w]+|[^\w]+$", "", text)
    return text or None

for field in TEXT_FIELDS:
    comments_long[f"{field}_norm"] = comments_long[field].apply(normalize_text)

comments_long["missing_expected_standard"] = comments_long["expected_standard_norm"].isna()
comments_long["missing_observed_issue"] = comments_long["observed_issue_norm"].isna()
comments_long["missing_suggested_fix"] = comments_long["suggested_fix_norm"].isna()

comments_long[[
    "comment_row_id",
    "screen_id",
    "screen_task_id",
    "comment_label",
    "comment_source",
    "observed_issue",
    "observed_issue_norm",
]].head(5)

,comment_row_id,screen_id,screen_task_id,comment_label,comment_source,observed_issue,observed_issue_norm
0,1,15,15_T01,Comment 1,Human,the text (Workouts) is small even when it is heading.,the text (workouts) is small even when it is heading
1,2,15,15_T01,Comment 2,Human,text (Plans) is in light red color on white background which is not making a good contrast.,text (plans) is in light red color on white background which is not making a good contrast
2,3,15,15_T01,Comment 3,Human,the back button size is small.,the back button size is small
3,4,15,15_T01,Comment 4,Human,the texts are disappearing at the bottom edge of the layout leaving no marginal space which is making it difficult for users to know the complete information,the texts are disappearing at the bottom edge of the layout leaving no marginal space which is making it difficult for users to know the complete information
4,5,15,15_T01,Comment 5,Human,Icon is not clearly visible,icon is not clearly visible


### 1.5 Apply Cleaning Rules

Hard removals are logged in an audit table rather than being silently discarded.

In [8]:
audit_parts = []
working_df = comments_long.copy()

def move_to_audit(df, mask, reason):
    removed = df.loc[mask].copy()
    if len(removed):
        removed["audit_reason"] = reason
    kept = df.loc[~mask].copy()
    return kept, removed

if DROP_LLM:
    working_df, removed = move_to_audit(working_df, working_df["comment_source"].ne("Human"), "non_human_comment")
    audit_parts.append(removed)

if DROP_MISSING_OBSERVED:
    working_df, removed = move_to_audit(working_df, working_df["missing_observed_issue"], "missing_observed_issue")
    audit_parts.append(removed)

if DROP_MISSING_EXPECTED:
    working_df, removed = move_to_audit(working_df, working_df["missing_expected_standard"], "missing_expected_standard")
    audit_parts.append(removed)

if DROP_MISSING_FIX:
    working_df, removed = move_to_audit(working_df, working_df["missing_suggested_fix"], "missing_suggested_fix")
    audit_parts.append(removed)

removed_comments_df = pd.concat(audit_parts, ignore_index=True) if audit_parts else pd.DataFrame(columns=list(working_df.columns) + ["audit_reason"])

print("Remaining after hard filters:", len(working_df))
print("Removed by hard filters:", len(removed_comments_df))
removed_comments_df["audit_reason"].value_counts(dropna=False)

Remaining after hard filters: 8272
Removed by hard filters: 3093


audit_reason
non_human_comment         3088
missing_observed_issue       5
Name: count, dtype: int64

### 1.6 Deduplicate Within The Same Task

First, drop exact duplicate triplets within the same `screen_task_id` using normalized
`expected_standard`, `observed_issue`, and `suggested_fix`.

For rows that still repeat only on normalized `observed_issue`, review them manually one group at a time
and choose which row to keep.


In [9]:
TASK_TRIPLET_KEY = [
    "screen_task_id",
    "expected_standard_norm",
    "observed_issue_norm",
    "suggested_fix_norm",
]

task_level_df = working_df.copy().sort_values(["screen_task_id", "comment_label", "comment_row_id"], kind="stable")

# Step 1: exact duplicate triplets within task -> keep one row only.
task_level_df["duplicate_task_triplet"] = task_level_df.duplicated(subset=TASK_TRIPLET_KEY, keep="first")
task_triplet_duplicates = task_level_df[task_level_df["duplicate_task_triplet"]].copy()
task_triplet_duplicates["audit_reason"] = "duplicate_task_triplet"

task_after_triplet = task_level_df[~task_level_df["duplicate_task_triplet"]].copy()

# Step 2: same observed issue still repeated within task -> manual review only.
task_duplicate_review = task_after_triplet[
    task_after_triplet.duplicated(subset=TASK_DEDUP_KEY, keep=False)
].copy().sort_values(["screen_task_id", "observed_issue_norm", "comment_row_id"], kind="stable")

task_duplicate_groups = list(task_duplicate_review.groupby(TASK_DEDUP_KEY, dropna=False).groups.keys())

print("Task-level rows before dedup:", len(task_level_df))
print("Exact triplet duplicates removed within task:", len(task_triplet_duplicates))
print("Observed-issue duplicate groups for manual review within task:", len(task_duplicate_groups))

# Temporary default until you manually resolve observed-issue duplicate groups.
task_level_clean = task_after_triplet.copy()


Task-level rows before dedup: 8272
Exact triplet duplicates removed within task: 47
Observed-issue duplicate groups for manual review within task: 22


In [10]:
def show_task_duplicate_group(group_idx):
    if group_idx < 0 or group_idx >= len(task_duplicate_groups):
        raise IndexError(f"group_idx must be between 0 and {len(task_duplicate_groups) - 1}")

    key = task_duplicate_groups[group_idx]
    screen_task_id, observed_issue_norm = key
    group_df = task_duplicate_review[
        (task_duplicate_review["screen_task_id"] == screen_task_id)
        & (task_duplicate_review["observed_issue_norm"] == observed_issue_norm)
    ].copy()

    print(f"Task duplicate group {group_idx + 1}/{len(task_duplicate_groups)}")
    print("screen_task_id:", screen_task_id)
    print("observed_issue_norm:", observed_issue_norm)
    return group_df[[
        "comment_row_id",
        "screen_id",
        "screen_task_id",
        "comment_label",
        "expected_standard",
        "observed_issue",
        "suggested_fix",
        "missing_parts",
    ]]

def apply_task_manual_choices(keep_comment_row_ids):
    if len(task_duplicate_groups) == 0:
        return task_after_triplet.copy()

    if len(keep_comment_row_ids) != len(task_duplicate_groups):
        raise ValueError(f"Provide exactly one comment_row_id per task duplicate group ({len(task_duplicate_groups)} total).")

    chosen = set(keep_comment_row_ids)
    duplicate_mask = task_after_triplet.duplicated(subset=TASK_DEDUP_KEY, keep=False)
    unresolved_group_rows = task_after_triplet.loc[duplicate_mask, "comment_row_id"]

    if not set(keep_comment_row_ids).issubset(set(unresolved_group_rows)):
        raise ValueError("All chosen comment_row_id values must come from task duplicate groups.")

    kept_duplicate_rows = task_after_triplet[duplicate_mask & task_after_triplet["comment_row_id"].isin(chosen)].copy()
    if len(kept_duplicate_rows) != len(task_duplicate_groups):
        raise ValueError("Your selections do not resolve to exactly one kept row per task duplicate group.")

    return pd.concat([
        task_after_triplet[~duplicate_mask].copy(),
        kept_duplicate_rows,
    ], ignore_index=True).sort_values(["screen_task_id", "comment_label", "comment_row_id"], kind="stable").reset_index(drop=True)

print("Use show_task_duplicate_group(i) to inspect one group at a time.")
print("Then run: task_level_clean = apply_task_manual_choices([...])")
print("Example first group:")
show_task_duplicate_group(21) if task_duplicate_groups else print("No task-level observed-issue duplicate groups to review.")


Use show_task_duplicate_group(i) to inspect one group at a time.
Then run: task_level_clean = apply_task_manual_choices([...])
Example first group:
Task duplicate group 22/22
screen_task_id: 9520_T01
observed_issue_norm: the text is difficult to read because it is too small and there is not enough contrast between the text and the background


,comment_row_id,screen_id,screen_task_id,comment_label,expected_standard,observed_issue,suggested_fix,missing_parts
1554,1555,9520,9520_T01,Comment 2,the text should be easy to read.,the text is difficult to read because it is too small and there is not enough contrast between the text and the background.,the text should be made larger and the contrast between the text and the background should be increased.,[]
1556,1557,9520,9520_T01,Comment 4,the text should be easy to read and respect rules of typography.,the text is difficult to read because it is too small and there is not enough contrast between the text and the background.,the text should be increased in size and the contrast between the text and the background should be increased.,[]


In [11]:
# After manually inspecting the groups, most parts were very similar with slight variations in wording. I chose the mre rich/representative comment from each group

task_level_clean = apply_task_manual_choices([277,2578,2580,382,3524,3605,4024,4572,5400,5675,6273,6625,7188,7930,825,8399,9220,9570,10230,11218,1524,1557])

### 1.7 Deduplicate Within The Same Screen

After task-level manual decisions are applied, repeat the same process at screen level:
drop exact duplicate triplets automatically, then review repeated normalized `observed_issue` groups
one at a time and choose which row to keep.


In [12]:
SCREEN_TRIPLET_KEY = [
    "screen_id",
    "expected_standard_norm",
    "observed_issue_norm",
    "suggested_fix_norm",
]

screen_input = task_level_clean.copy().sort_values(["screen_id", "screen_task_id", "comment_label", "comment_row_id"], kind="stable")

# Step 1: exact duplicate triplets within screen -> keep one row only.
screen_input["duplicate_screen_triplet"] = screen_input.duplicated(subset=SCREEN_TRIPLET_KEY, keep="first")
screen_triplet_duplicates = screen_input[screen_input["duplicate_screen_triplet"]].copy()
screen_triplet_duplicates["audit_reason"] = "duplicate_screen_triplet"

screen_after_triplet = screen_input[~screen_input["duplicate_screen_triplet"]].copy()

# Step 2: same observed issue still repeated across tasks in the same screen -> manual review only.
screen_duplicate_review = screen_after_triplet[
    screen_after_triplet.duplicated(subset=SCREEN_DEDUP_KEY, keep=False)
].copy().sort_values(["screen_id", "observed_issue_norm", "screen_task_id", "comment_row_id"], kind="stable")

screen_duplicate_groups = list(screen_duplicate_review.groupby(SCREEN_DEDUP_KEY, dropna=False).groups.keys())

print("Screen-level rows before dedup:", len(screen_input))
print("Exact triplet duplicates removed within screen:", len(screen_triplet_duplicates))
print("Observed-issue duplicate groups for manual review within screen:", len(screen_duplicate_groups))

# Temporary default until you manually resolve observed-issue duplicate groups.
human_comments_screen_reference = screen_after_triplet.copy()
human_comments_screen_reference = human_comments_screen_reference.sort_values(["screen_id", "observed_issue_norm"], kind="stable").reset_index(drop=True)
human_comments_screen_reference.insert(
    0,
    "canonical_comment_id",
    [f"S{row.screen_id}_C{i+1:03d}" for i, row in human_comments_screen_reference.reset_index().iterrows()],
)


Screen-level rows before dedup: 8203
Exact triplet duplicates removed within screen: 215
Observed-issue duplicate groups for manual review within screen: 31


In [13]:
def show_screen_duplicate_group(group_idx):
    if group_idx < 0 or group_idx >= len(screen_duplicate_groups):
        raise IndexError(f"group_idx must be between 0 and {len(screen_duplicate_groups) - 1}")

    key = screen_duplicate_groups[group_idx]
    screen_id, observed_issue_norm = key
    group_df = screen_duplicate_review[
        (screen_duplicate_review["screen_id"] == screen_id)
        & (screen_duplicate_review["observed_issue_norm"] == observed_issue_norm)
    ].copy()

    print(f"Screen duplicate group {group_idx + 1}/{len(screen_duplicate_groups)}")
    print("screen_id:", screen_id)
    print("observed_issue_norm:", observed_issue_norm)
    return group_df[[
        "comment_row_id",
        "screen_id",
        "screen_task_id",
        "comment_label",
        "expected_standard",
        "observed_issue",
        "suggested_fix",
        "missing_parts",
    ]]

def apply_screen_manual_choices(keep_comment_row_ids):
    if len(screen_duplicate_groups) == 0:
        base_df = screen_after_triplet.copy()
    else:
        if len(keep_comment_row_ids) != len(screen_duplicate_groups):
            raise ValueError(f"Provide exactly one comment_row_id per screen duplicate group ({len(screen_duplicate_groups)} total).")

        chosen = set(keep_comment_row_ids)
        duplicate_mask = screen_after_triplet.duplicated(subset=SCREEN_DEDUP_KEY, keep=False)
        unresolved_group_rows = screen_after_triplet.loc[duplicate_mask, "comment_row_id"]

        if not set(keep_comment_row_ids).issubset(set(unresolved_group_rows)):
            raise ValueError("All chosen comment_row_id values must come from screen duplicate groups.")

        kept_duplicate_rows = screen_after_triplet[duplicate_mask & screen_after_triplet["comment_row_id"].isin(chosen)].copy()
        if len(kept_duplicate_rows) != len(screen_duplicate_groups):
            raise ValueError("Your selections do not resolve to exactly one kept row per screen duplicate group.")

        base_df = pd.concat([
            screen_after_triplet[~duplicate_mask].copy(),
            kept_duplicate_rows,
        ], ignore_index=True)

    base_df = base_df.sort_values(["screen_id", "observed_issue_norm", "screen_task_id", "comment_row_id"], kind="stable").reset_index(drop=True)
    base_df = base_df.copy()
    if "canonical_comment_id" in base_df.columns:
        base_df = base_df.drop(columns=["canonical_comment_id"])
    base_df.insert(0, "canonical_comment_id", [f"S{row.screen_id}_C{i+1:03d}" for i, row in base_df.reset_index().iterrows()])
    return base_df

print("Use show_screen_duplicate_group(i) to inspect one group at a time.")
print("Then run: human_comments_screen_reference = apply_screen_manual_choices([...])")
print("Example first group:")
show_screen_duplicate_group(30) if screen_duplicate_groups else print("No screen-level observed-issue duplicate groups to review.")

Use show_screen_duplicate_group(i) to inspect one group at a time.
Then run: human_comments_screen_reference = apply_screen_manual_choices([...])
Example first group:
Screen duplicate group 31/31
screen_id: 66639
observed_issue_norm: the design does not create a sense of balance and order, which makes it look chaotic and disorganized


,comment_row_id,screen_id,screen_task_id,comment_label,expected_standard,observed_issue,suggested_fix,missing_parts
7172,10567,66639,66639_T01,Comment 8,the design should create a sense of balance and order.,"the design does not create a sense of balance and order, which makes it look chaotic and disorganized.",the designer should use more white space and align the elements along a horizontal or vertical axis.,[]
7178,10573,66639,66639_T02,Comment 5,the design should create a sense of balance and order.,"the design does not create a sense of balance and order, which makes it look chaotic and disorganized.",the designer should use less white space and align the elements along a horizontal or vertical axis.,[]


In [14]:
# After manually inspecting the groups, most parts were very similar with slight variations in wording. I chose the mre rich/representative comment from each group
human_comments_screen_reference = apply_screen_manual_choices([389,819,1047,1239,2591,2746,2760,2766,2920,3287,3440,3614,3662,3875,4641,5503,5515,5943,5992,6788,7429,7781,8050, 8405,8480,8487,9189,9388,9722,10165,10567])

### 1.8 Build Final Task-Shaped Output

The final saved table is reshaped back to one row per task with a `comments` list of dictionaries:
`screen_id`, `app_category`, `screen_task_id`, `task`, `comments`.


In [ ]:
task_metadata = (
    uicrit_df[["screen_id", "app_category", "screen_task_id", "task"]]
    .drop_duplicates(subset=["screen_task_id"])
    .copy()
)

comment_payload_cols = [
    "comment_label",
    "missing_parts",
    "expected_standard",
    "observed_issue",
    "suggested_fix",
    "bounding_box",
    "raw_text",
]

final_exact_dedup_comments_df = (
    human_comments_screen_reference
    .sort_values(["screen_id", "screen_task_id", "comment_label"], kind="stable")
    .groupby(["screen_id", "screen_task_id", "task"], sort=True)[comment_payload_cols]
    .apply(lambda x: x.to_dict(orient="records"))
    .reset_index(name="comments")
)

final_exact_dedup_comments_df = (
    task_metadata.merge(final_exact_dedup_comments_df, on=["screen_id", "screen_task_id", "task"], how="left")
    .sort_values(["screen_id", "screen_task_id"], kind="stable")
    .reset_index(drop=True)
)

final_exact_dedup_comments_df["comments"] = final_exact_dedup_comments_df["comments"].apply(lambda x: x if isinstance(x, list) else [])
final_comments_before_empty_drop = final_exact_dedup_comments_df.copy()

removed_empty_comment_tasks = int((final_comments_before_empty_drop["comments"].apply(len) == 0).sum())
original_num_screen_tasks = len(final_comments_before_empty_drop)

final_exact_dedup_comments_df = final_comments_before_empty_drop[final_comments_before_empty_drop["comments"].apply(len) > 0].copy().reset_index(drop=True)
final_exact_dedup_comments_df = final_exact_dedup_comments_df[["screen_id", "app_category", "screen_task_id", "task", "comments"]]

print("Original screen_task rows:", original_num_screen_tasks)
print("Removed screen_task rows with empty comments:", removed_empty_comment_tasks)
print("Final screen_task rows:", len(final_exact_dedup_comments_df))

final_exact_dedup_comments_df.head(3)

Original screen_task rows: 2957
Removed screen_task rows with empty comments: 970
Final screen_task rows: 1987


,screen_id,app_category,screen_task_id,task,comments
0,15,Health & Fitness,15_T01,Plan and Start Full Body Workouts,"[{'comment_label': 'Comment 1', 'missing_parts': [], 'expected_standard': 'the text’s visual treatment and formatting should make it easy to understand.', '..."
1,15,Health & Fitness,15_T02,Tap on the Plus icon or explore the Workout Plans.,"[{'comment_label': 'Comment 1', 'missing_parts': [], 'expected_standard': 'make the most important information visually dominant.', 'observed_issue': 'the b..."
2,28,Finance,28_T02,Enter details to sign in or click on activate mobile banking/Need help signing in.,"[{'comment_label': 'Comment 1', 'missing_parts': [], 'expected_standard': 'have high contrast, clear text size for optimal readability.', 'observed_issue': ..."


#### Final Cleaning Check

Inspect which screen-task rows are removed at the final step because their `comments` list is empty.
Interpretation of why they became empty should rely on the earlier filtering and audit tables.


In [ ]:
final_cleaning_check = final_comments_before_empty_drop[["screen_id", "app_category", "screen_task_id", "task", "comments"]].copy()
final_cleaning_check["num_comments"] = final_cleaning_check["comments"].apply(len)

removed_empty_comment_tasks_df = final_cleaning_check[final_cleaning_check["num_comments"] == 0].copy()

print("Original screen-task rows before final empty-comments drop:", original_num_screen_tasks)
print("Removed screen-task rows with empty comments:", removed_empty_comment_tasks)
print("Final retained screen-task rows:", len(final_exact_dedup_comments_df))

display(removed_empty_comment_tasks_df.head(10))


Original screen-task rows before final empty-comments drop: 2957
Removed screen-task rows with empty comments: 970
Final retained screen-task rows: 1987


,screen_id,app_category,screen_task_id,task,comments,num_comments
2,15,Health & Fitness,15_T03,select and get started with Full body workouts,[],0
3,28,Finance,28_T01,Enter details to Sing In to Scotiabank.,[],0
8,67,Education,67_T03,View/ Upload photo.,[],0
12,193,Social,193_T03,Search members to make new friends,[],0
13,233,Weather,233_T01,Change Map Settings.,[],0
18,288,Social,288_T03,Select your preferences.,[],0
21,342,None,342_T03,Send Feedback about Great Sword.,[],0
24,422,Social,422_T03,sign-in as Laura and edit the details,[],0
26,445,Medical,445_T02,Select to add the premium feature,[],0
29,640,Music & Audio,640_T02,Choose options to listen FM radio stations,[],0


### 1.9 Validation Checks

These checks confirm that the final task-shaped table is ready to save.


In [ ]:
assert (task_level_clean["comment_source"] == "Human").all(), "Non-human comment survived task-level cleaning."
assert human_comments_screen_reference["observed_issue"].notna().all(), "Screen-level reference contains missing observed issue."
assert not human_comments_screen_reference.duplicated(subset=SCREEN_DEDUP_KEY).any(), "Screen-level observed-issue duplicates still remain. Resolve them manually first."
assert final_exact_dedup_comments_df.columns.tolist() == ["screen_id", "app_category", "screen_task_id", "task", "comments"], "Final output columns do not match the expected schema."
assert final_exact_dedup_comments_df["screen_task_id"].is_unique, "Final output should have one row per screen_task_id."
assert final_exact_dedup_comments_df["comments"].apply(len).gt(0).all(), "Final output still contains empty comments lists."

print("All validation checks passed.")


All validation checks passed.


### 1.10 Summary

Summarize both what was dropped during cleaning and what remains in the final dataset.


In [ ]:
final_comment_counts = final_exact_dedup_comments_df["comments"].apply(len)

final_comments_long = final_exact_dedup_comments_df[["screen_id", "app_category", "screen_task_id", "task", "comments"]].explode("comments", ignore_index=True)
final_comments_long = final_comments_long[final_comments_long["comments"].notna()].copy()

if len(final_comments_long):
    final_comment_fields = pd.json_normalize(final_comments_long["comments"])
    final_comments_long = pd.concat([final_comments_long.drop(columns=["comments"]).reset_index(drop=True), final_comment_fields.reset_index(drop=True)], axis=1)
else:
    final_comments_long = pd.DataFrame(columns=["screen_id", "app_category", "screen_task_id", "task"])

drop_summary = pd.DataFrame([
    {"metric": "removed_non_human_comments", "value": int((removed_comments_df["audit_reason"] == "non_human_comment").sum())},
    {"metric": "removed_missing_observed_issue", "value": int((removed_comments_df["audit_reason"] == "missing_observed_issue").sum())},
    {"metric": "removed_duplicate_task_triplet", "value": int(len(task_triplet_duplicates))},
    {"metric": "removed_duplicate_screen_triplet", "value": int(len(screen_triplet_duplicates))},
    {"metric": "task_duplicate_groups_manually_reviewed", "value": int(len(task_duplicate_groups))},
    {"metric": "screen_duplicate_groups_manually_reviewed", "value": int(len(screen_duplicate_groups))},
])

display(drop_summary)


,metric,value
0,removed_non_human_comments,3088
1,removed_missing_observed_issue,5
2,removed_duplicate_task_triplet,47
3,removed_duplicate_screen_triplet,215
4,task_duplicate_groups_manually_reviewed,22
5,screen_duplicate_groups_manually_reviewed,31


In [ ]:
num_missing_expected = final_comments_long["expected_standard"].isna().sum() if "expected_standard" in final_comments_long.columns else 0
num_missing_fix = final_comments_long["suggested_fix"].isna().sum() if "suggested_fix" in final_comments_long.columns else 0
num_missing_both = ((final_comments_long["expected_standard"].isna()) & (final_comments_long["suggested_fix"].isna())).sum() if len(final_comments_long) else 0

descriptive_summary = pd.DataFrame([
    {"metric": "original_num_screen_tasks", "value": int(original_num_screen_tasks)},
    {"metric": "removed_empty_comment_screen_tasks", "value": int(removed_empty_comment_tasks)},
    {"metric": "final_num_screen_tasks", "value": int(len(final_exact_dedup_comments_df))},
    {"metric": "num_screens", "value": int(final_exact_dedup_comments_df["screen_id"].nunique())},
    {"metric": "num_final_comments", "value": int(len(final_comments_long))},
    {"metric": "avg_comments_per_task", "value": float(final_comment_counts.mean())},
    {"metric": "median_comments_per_task", "value": float(final_comment_counts.median())},
    {"metric": "missing_expected_standard", "value": int(num_missing_expected)},
    {"metric": "missing_suggested_fix", "value": int(num_missing_fix)},
    {"metric": "missing_expected_and_fix", "value": int(num_missing_both)},
])

display(descriptive_summary.round(0))
print("Comments per retained task distribution:")
print(final_comment_counts.describe().round(2))


,metric,value
0,original_num_screen_tasks,2957.0
1,removed_empty_comment_screen_tasks,970.0
2,final_num_screen_tasks,1987.0
3,num_screens,1000.0
4,num_final_comments,7957.0
5,avg_comments_per_task,4.0
6,median_comments_per_task,4.0
7,missing_expected_standard,4.0
8,missing_suggested_fix,6.0
9,missing_expected_and_fix,1.0


Comments per retained task distribution:
count    1987.00
mean        4.00
std         2.06
min         1.00
25%         2.00
50%         4.00
75%         5.00
max        13.00
Name: comments, dtype: float64


### 1.11 Save Outputs

Save only the parquet outputs needed for downstream analysis.


In [ ]:
final_exact_dedup_df_path = DATA_DIR / "human_comments_exact_dedup.parquet"
audit_path = OUTPUT_DIR / "human_comments_exact_dedup_audit_log.parquet"

final_exact_dedup_comments_df.to_parquet(final_exact_dedup_df_path, index=False)
removed_comments_df.to_parquet(audit_path, index=False)

print("Saved:")
for path in [final_exact_dedup_df_path, audit_path]:
    print(path)

Saved:
cleaned_dataset\human_comments_exact_dedup.parquet
cleaned_dataset\human_critiques\human_comments_exact_dedup_audit_log.parquet


## 2. Semantic Duplicates Within The Same Screen

Wworkflow:
- generate candidate comment pairs only within the same `screen_id`
- first score candidate pairs with embeddings + cosine similarity using a relatively high threshold
- manually inspect a sample of high-similarity pairs to tune the threshold
- give a summary of the duplications
- send the remaining candidate pairs to OpenAI Batch for a stricter same-meaning judgment
- keep one canonical row per semantic group after review


### 2.0 Imports And Setup

Load the shared semantic-matching utilities and batch helpers used throughout Section 2.


In [1]:
import sys
from collections import defaultdict

from openai import OpenAI

PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.append(str(PROJECT_ROOT / "code"))

from utils.comment_matching_utils import (
    DEFAULT_EMBEDDING_MODEL_NAME,
    DEFAULT_REVIEW_MODEL,
    build_standard_preview_df,
    build_standardized_human_comments_from_nested,
    build_standardized_pairs,
    identify_review_needed,
    inspect_manual_review_rows,
    finalize_review_decisions,
    load_embedding_model,
    parse_semantic_review_results_jsonl,
    prepare_review_chunks,
    score_standardized_pairs,
    standardize_comment_table,
    write_jsonl,
)
from utils.openai_batch_manager import OpenAIBatchManager

NameError: name 'Path' is not defined

### 2.1 Build Comment-Level Input

Start from `final_exact_dedup_comments_df` and explode the `comments` column so each cleaned human comment is one row.
Keep the screen/task identifiers needed to map decisions back later.


In [ ]:
final_exact_dedup_comments_df = pd.read_parquet(final_exact_dedup_df_path)

semantic_comments_long = build_standardized_human_comments_from_nested(
    final_exact_dedup_comments_df[["screen_id", "app_category", "screen_task_id", "task", "comments"]].copy(),
    nested_comments_col='comments',
    comment_id_prefix='H',
)

print("Comment-level rows for semantic dedup:", len(semantic_comments_long))
print("Screens represented:", semantic_comments_long["screen_id"].nunique())
semantic_comments_long.head(3)

Comment-level rows for semantic dedup: 7957
Screens represented: 1000


,comment_id,source_type,screen_id,screen_task_id,task,app_category,observed_issue,expected_standard,suggested_fix,match_text,missing_parts,raw_text,bounding_box,comment_label
0,H_000001,human,15,15_T01,Plan and Start Full Body Workouts,Health & Fitness,the text (Workouts) is small even when it is heading.,the text’s visual treatment and formatting should make it easy to understand.,increase font size and weight to make it look like a heading,the text (Workouts) is small even when it is heading.,[],"Comment 1 The expected standard is that the text’s visual treatment and formatting should make it easy to understand. In the current design, the text (Worko...","{'x_max': 0.40545597, 'x_min': 0.1559446, 'y_max': 0.09649163, 'y_min': 0.05137866}",Comment 1
1,H_000002,human,15,15_T01,Plan and Start Full Body Workouts,Health & Fitness,text (Plans) is in light red color on white background which is not making a good contrast.,the text and background colors used in the design should be complementary and easy to read.,change colors to be more complementary to each other (change texts to dark colors) to make it a good contrast and easier to read.,text (Plans) is in light red color on white background which is not making a good contrast.,[],"Comment 2 The expected standard is that the text and background colors used in the design should be complementary and easy to read. In the current design, t...","{'x_max': 0.1559446, 'x_min': 0.02896114, 'y_max': 0.18045189, 'y_min': 0.15037657}",Comment 2
2,H_000003,human,15,15_T01,Plan and Start Full Body Workouts,Health & Fitness,the back button size is small.,the design should make the most important information visually dominant.,increase the back button size to make it visually prominent for the users.,the back button size is small.,[],"Comment 3 The expected standard is that the design should make the most important information visually dominant. In the current design, the back button size...","{'x_max': 0.1002501, 'x_min': 0.03787226, 'y_max': 0.09147908, 'y_min': 0.05513808}",Comment 3


### 2.2 Generate Within-Screen Candidate Pairs

Create candidate pairs only within the same `screen_id`.
Do not compare comments across different screens.


In [115]:
semantic_candidate_pairs = build_standardized_pairs(
    semantic_comments_long,
    semantic_comments_long,
    pair_id_prefix='PAIR',
    exclude_same_comment_id_pairs=True,
)
# keep only one direction for human-human pairs
semantic_candidate_pairs = semantic_candidate_pairs[
    semantic_candidate_pairs['left_comment_id'] < semantic_candidate_pairs['right_comment_id']
].copy().reset_index(drop=True)

print("Within-screen candidate pairs:", len(semantic_candidate_pairs))
print("Screens with at least one pair:", semantic_candidate_pairs["screen_id"].nunique() if len(semantic_candidate_pairs) else 0)

if len(semantic_candidate_pairs):
    pairs_per_screen = semantic_candidate_pairs.groupby("screen_id").size()
    print("Candidate pairs per screen:")
    print(pairs_per_screen.describe().round(2))

semantic_candidate_pairs.head(5)

Within-screen candidate pairs: 31902
Screens with at least one pair: 999
Candidate pairs per screen:
count    999.00
mean      31.93
std       24.28
min        1.00
25%       15.00
50%       28.00
75%       45.00
max      171.00
dtype: float64


,pair_id,screen_id,app_category,left_comment_id,left_source_type,left_screen_task_id,left_task,left_observed_issue,left_expected_standard,left_suggested_fix,left_match_text,missing_parts_left,raw_text_left,bounding_box_left,comment_label_left,right_comment_id,right_source_type,right_screen_task_id,right_task,right_observed_issue,right_expected_standard,right_suggested_fix,right_match_text,missing_parts_right,raw_text_right,bounding_box_right,comment_label_right
0,PAIR_0000001,15,Health & Fitness,H_000001,human,15_T01,Plan and Start Full Body Workouts,the text (Workouts) is small even when it is heading.,the text’s visual treatment and formatting should make it easy to understand.,increase font size and weight to make it look like a heading,the text (Workouts) is small even when it is heading.,[],"Comment 1 The expected standard is that the text’s visual treatment and formatting should make it easy to understand. In the current design, the text (Worko...","{'x_max': 0.40545597, 'x_min': 0.1559446, 'y_max': 0.09649163, 'y_min': 0.05137866}",Comment 1,H_000002,human,15_T01,Plan and Start Full Body Workouts,text (Plans) is in light red color on white background which is not making a good contrast.,the text and background colors used in the design should be complementary and easy to read.,change colors to be more complementary to each other (change texts to dark colors) to make it a good contrast and easier to read.,text (Plans) is in light red color on white background which is not making a good contrast.,[],"Comment 2 The expected standard is that the text and background colors used in the design should be complementary and easy to read. In the current design, t...","{'x_max': 0.1559446, 'x_min': 0.02896114, 'y_max': 0.18045189, 'y_min': 0.15037657}",Comment 2
1,PAIR_0000002,15,Health & Fitness,H_000001,human,15_T01,Plan and Start Full Body Workouts,the text (Workouts) is small even when it is heading.,the text’s visual treatment and formatting should make it easy to understand.,increase font size and weight to make it look like a heading,the text (Workouts) is small even when it is heading.,[],"Comment 1 The expected standard is that the text’s visual treatment and formatting should make it easy to understand. In the current design, the text (Worko...","{'x_max': 0.40545597, 'x_min': 0.1559446, 'y_max': 0.09649163, 'y_min': 0.05137866}",Comment 1,H_000003,human,15_T01,Plan and Start Full Body Workouts,the back button size is small.,the design should make the most important information visually dominant.,increase the back button size to make it visually prominent for the users.,the back button size is small.,[],"Comment 3 The expected standard is that the design should make the most important information visually dominant. In the current design, the back button size...","{'x_max': 0.1002501, 'x_min': 0.03787226, 'y_max': 0.09147908, 'y_min': 0.05513808}",Comment 3
2,PAIR_0000003,15,Health & Fitness,H_000001,human,15_T01,Plan and Start Full Body Workouts,the text (Workouts) is small even when it is heading.,the text’s visual treatment and formatting should make it easy to understand.,increase font size and weight to make it look like a heading,the text (Workouts) is small even when it is heading.,[],"Comment 1 The expected standard is that the text’s visual treatment and formatting should make it easy to understand. In the current design, the text (Worko...","{'x_max': 0.40545597, 'x_min': 0.1559446, 'y_max': 0.09649163, 'y_min': 0.05137866}",Comment 1,H_000004,human,15_T01,Plan and Start Full Body Workouts,the texts are disappearing at the bottom edge of the layout leaving no marginal space which is making it difficult for users to know the complete information,design should be well organized.,redesign the UI to fit the elements within the page layout.,the texts are disappearing at the bottom edge of the layout leaving no marginal space which is making it difficult for users to know the complete information,[],"Comment 4 The expe

### 2.3 Embeddings And Cosine Similarity

Embed each comment text, compute cosine similarity for the within-screen pairs, and keep only pairs above a relatively high threshold.
Use this as a conservative semantic-duplicate candidate stage, not as the final decision.


In [ ]:
EMBEDDING_MODEL_NAME = DEFAULT_EMBEDDING_MODEL_NAME
SEMANTIC_SIMILARITY_THRESHOLD = 0.88

embedding_model = load_embedding_model(EMBEDDING_MODEL_NAME)
semantic_candidate_pairs, embeddings_df = score_standardized_pairs(
    semantic_candidate_pairs,
    semantic_comments_long,
    embedding_model=embedding_model,
)

semantic_high_similarity_pairs = semantic_candidate_pairs[
    semantic_candidate_pairs["cosine_similarity"] >= SEMANTIC_SIMILARITY_THRESHOLD
].copy().sort_values(["cosine_similarity", "screen_id"], ascending=[False, True], kind="stable")

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Embedding rows:", len(embeddings_df))
print("Candidate pairs scored:", len(semantic_candidate_pairs))
print("Pairs above threshold:", len(semantic_high_similarity_pairs))
print("Threshold:", SEMANTIC_SIMILARITY_THRESHOLD)

display(build_standard_preview_df(semantic_high_similarity_pairs, include_score=True, limit=10, sort_cols=["cosine_similarity", "screen_id"], ascending=[False, True]))

In [ ]:
display(
    build_standard_preview_df(
        semantic_high_similarity_pairs,
        include_score=True,
        sort_cols=["cosine_similarity", "screen_id"],
        ascending=[False, True],
        limit=10,
    ).tail(10)
)

### 2.4 Manual Threshold Check

Inspect a sample of high-similarity pairs manually before locking the threshold.
Tune the threshold so it favors precision over recall.


In [ ]:
print("Cosine similarity distribution:")
print(semantic_candidate_pairs["cosine_similarity"].describe().round(4))

display(build_standard_preview_df(semantic_high_similarity_pairs, include_score=True, limit=10).tail(10))


In [ ]:
SEMANTIC_SIMILARITY_THRESHOLD = 0.88
semantic_high_similarity_pairs = semantic_candidate_pairs[
    semantic_candidate_pairs["cosine_similarity"] >= SEMANTIC_SIMILARITY_THRESHOLD
].copy().sort_values(
    ["cosine_similarity", "screen_id"], ascending=[False, True], kind="stable"
)

print("Cosine similarity distribution:")
print(semantic_candidate_pairs["cosine_similarity"].describe().round(4))
display(build_standard_preview_df(semantic_high_similarity_pairs, include_score=True, limit=15))


### 2.5 Clean Semantic Duplicates By Threshold

This step applies the selected cosine-similarity threshold to automatically remove strong semantic duplicates within the same screen.

The process is:

1. Treat each high-similarity pair as a semantic-duplicate link within a screen.
2. Build semantic groups from the linked comments.
3. For each group, keep one representative comment.
4. Choose the representative as the comment with the longer available text (`observed_issue`).
5. Drop the other comments in the same semantic group.
6. Rebuild the task-shaped dataframe after this semantic deduplication.

Note:
The current grouping logic uses linked pairs to form groups, so if A is linked to B and B is linked to C, all three may be grouped together.

In [ ]:
semantic_comments_for_grouping = semantic_comments_long.copy()
semantic_comments_for_grouping["keep_text"] = (
    semantic_comments_for_grouping["observed_issue"]
    .fillna(semantic_comments_for_grouping["raw_text"])
    .fillna("")
)
semantic_comments_for_grouping["keep_text_length"] = semantic_comments_for_grouping["keep_text"].astype(str).str.len()

semantic_group_rows = []
semantic_duplicate_edges = semantic_high_similarity_pairs.copy()

for screen_id, screen_comments in semantic_comments_for_grouping.groupby("screen_id", sort=True):
    screen_comment_ids = screen_comments["comment_id"].tolist()
    parent = {cid: cid for cid in screen_comment_ids}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    screen_edges = semantic_duplicate_edges[semantic_duplicate_edges["screen_id"] == screen_id]
    for _, edge in screen_edges.iterrows():
        union(edge["left_comment_id"], edge["right_comment_id"])

    groups = defaultdict(list)
    for cid in screen_comment_ids:
        groups[find(cid)].append(cid)

    for group_num, member_ids in enumerate(groups.values(), start=1):
        group_comments = screen_comments[screen_comments["comment_id"].isin(member_ids)].copy()
        group_comments = group_comments.sort_values(
            ["keep_text_length", "comment_id"],
            ascending=[False, True],
            kind="stable",
        )
        keep_id = group_comments.iloc[0]["comment_id"]
        semantic_group_id = f"SG_{screen_id}_{group_num:03d}"

        for _, row in group_comments.iterrows():
            semantic_group_rows.append({
                "semantic_group_id": semantic_group_id,
                "screen_id": screen_id,
                "comment_id": row["comment_id"],
                "keep_comment_id": keep_id,
                "keep_in_group": row["comment_id"] == keep_id,
                "group_size": len(group_comments),
                "keep_text_length": row["keep_text_length"],
            })

semantic_groups_df = pd.DataFrame(semantic_group_rows)
semantic_comments_semantic_dedup_long = semantic_comments_for_grouping.merge(
    semantic_groups_df,
    on=["screen_id", "comment_id"],
    how="left",
)
semantic_comments_semantic_dedup_long = semantic_comments_semantic_dedup_long[
    semantic_comments_semantic_dedup_long["keep_in_group"]
].copy()

semantic_comment_payload_cols = [
    "comment_label",
    "missing_parts",
    "expected_standard",
    "observed_issue",
    "suggested_fix",
    "bounding_box",
    "raw_text",
]

semantic_dedup_comments_df = (
    semantic_comments_semantic_dedup_long
    .sort_values(["screen_id", "screen_task_id", "comment_label"], kind="stable")
    .groupby(["screen_id", "screen_task_id", "task"], sort=True)[semantic_comment_payload_cols]
    .apply(lambda x: x.to_dict(orient="records"))
    .reset_index(name="comments")
)

semantic_dedup_comments_df = (
    task_metadata.merge(
        semantic_dedup_comments_df,
        on=["screen_id", "screen_task_id", "task"],
        how="left",
    )
    .sort_values(["screen_id", "screen_task_id"], kind="stable")
    .reset_index(drop=True)
)
semantic_dedup_comments_df["comments"] = semantic_dedup_comments_df["comments"].apply(lambda x: x if isinstance(x, list) else [])
semantic_dedup_comments_df = semantic_dedup_comments_df[semantic_dedup_comments_df["comments"].apply(len) > 0].copy().reset_index(drop=True)
semantic_dedup_comments_df = semantic_dedup_comments_df[["screen_id", "app_category", "screen_task_id", "task", "comments"]]

print("Pairs above threshold:", len(semantic_high_similarity_pairs))
print("Semantic groups formed:", semantic_groups_df["semantic_group_id"].nunique())
print("Comments before semantic dedup:", len(semantic_comments_long))
print("Comments after semantic dedup:", len(semantic_comments_semantic_dedup_long))
print("Task rows after semantic dedup:", len(semantic_dedup_comments_df))

semantic_groups_df.sort_values(["group_size", "screen_id"], ascending=[False, True], kind="stable").head(10)


### 2.6 Chunked OpenAI Batch Flow

Process the uncertain pairs in small chunks. Start with one chunk, inspect the returned decisions, and continue chunk by chunk as needed.


In [ ]:
BATCH_REVIEW_LOWER_BOUND = 0.50
BATCH_MODEL = DEFAULT_REVIEW_MODEL
BATCH_CHUNK_SIZE = 5
CURRENT_BATCH_CHUNK_INDEX = 0  # 0-based

semantic_batch_candidates = semantic_candidate_pairs[
    (semantic_candidate_pairs["cosine_similarity"] >= BATCH_REVIEW_LOWER_BOUND)
    & (semantic_candidate_pairs["cosine_similarity"] < SEMANTIC_SIMILARITY_THRESHOLD)
].copy().sort_values(["screen_id", "cosine_similarity"], ascending=[True, False], kind="stable").reset_index(drop=True)

semantic_batch_candidates["left_text"] = semantic_batch_candidates["left_observed_issue"]
semantic_batch_candidates["right_text"] = semantic_batch_candidates["right_observed_issue"]

semantic_batch_candidates["custom_id"] = semantic_batch_candidates.apply(
    lambda row: f"semantic_pair::{row['left_comment_id']}::{row['right_comment_id']}",
    axis=1,
)

semantic_batch_requests, semantic_batch_chunks, semantic_batch_candidate_chunks = prepare_review_chunks(
    semantic_batch_candidates,
    chunk_size=BATCH_CHUNK_SIZE,
    custom_id_col="custom_id",
    left_id_col="left_comment_id",
    right_id_col="right_comment_id",
    screen_id_col="screen_id",
    left_text_col="left_text",
    right_text_col="right_text",
    left_screen_task_id_col="left_screen_task_id",
    right_screen_task_id_col="right_screen_task_id",
    left_source_col="left_source_type",
    right_source_col="right_source_type",
    model=BATCH_MODEL,
)

print("Batch candidate pairs:", len(semantic_batch_candidates))
print("Chunk size:", BATCH_CHUNK_SIZE)
print("Number of chunks:", len(semantic_batch_chunks))


In [ ]:
CURRENT_BATCH_REQUESTS_PATH = OUTPUT_DIR / f"semantic_duplicate_batch_requests_chunk_{CURRENT_BATCH_CHUNK_INDEX:03d}.jsonl"
CURRENT_BATCH_RESULTS_PATH = OUTPUT_DIR / f"semantic_duplicate_batch_results_chunk_{CURRENT_BATCH_CHUNK_INDEX:03d}.jsonl"

current_chunk_requests = semantic_batch_chunks[CURRENT_BATCH_CHUNK_INDEX]
current_chunk_candidates = semantic_batch_candidate_chunks[CURRENT_BATCH_CHUNK_INDEX]

write_jsonl(current_chunk_requests, CURRENT_BATCH_REQUESTS_PATH)

print("Current chunk index:", CURRENT_BATCH_CHUNK_INDEX)
print("Requests in current chunk:", len(current_chunk_requests))
print("Chunk request path:", CURRENT_BATCH_REQUESTS_PATH)
display(build_standard_preview_df(current_chunk_candidates, include_score=True, limit=10))


#### 2.6.1 Run Current Chunk

Upload the current chunk, create the batch, monitor it, and retrieve the output.


In [ ]:
batch_client = OpenAI()
batch_manager = OpenAIBatchManager(batch_client, OUTPUT_DIR / "openai_semantic_batch", "semantic_duplicates")

current_input_file_id = batch_manager.upload_file(CURRENT_BATCH_REQUESTS_PATH)
current_batch = batch_manager.create_batch(current_input_file_id)
current_batch_id = getattr(current_batch, "id", None) if current_batch else None

print("Current input file id:", current_input_file_id)
print("Current batch id:", current_batch_id)


In [ ]:
state, current_batch_status = batch_manager.check_batch(current_batch_id, verbose=True)
print("Resolved state:", state)

In [ ]:
current_output_path = batch_manager.retrieve_batch_output(
    batch_id=current_batch_id,
    base_name=f"semantic_duplicate_batch_results_chunk_{CURRENT_BATCH_CHUNK_INDEX:03d}"
)
print("Current chunk batch results saved to:", current_output_path)

#### 2.6.2 Review Current Chunk Output

Inspect the returned decisions for the current chunk before moving to the next chunk.


In [ ]:
REVIEW_CONFIDENCE_THRESHOLD = 0.90

if current_output_path and Path(current_output_path).exists():
    current_chunk_results_df = parse_semantic_review_results_jsonl(
        current_output_path,
        current_chunk_candidates,
        left_id_col="left_comment_id",
        right_id_col="right_comment_id",
        include_candidate_columns=[
            "left_screen_task_id",
            "right_screen_task_id",
            "left_text",
            "right_text",
            "cosine_similarity",
        ],
    )

    display(current_chunk_results_df.head(10))

    review_needed_df = identify_review_needed(
        current_chunk_results_df,
        confidence_threshold=REVIEW_CONFIDENCE_THRESHOLD,
    )

    print("Rows needing interactive review in current chunk:", len(review_needed_df))
    display(review_needed_df)
else:
    print("Current chunk output file not found yet. Run the retrieval cell first.")


#### 2.6.3 Interactive Manual Review

Inspect `manual_review` rows one by one and decide on the spot which action to take.


In [ ]:
manual_review_decisions_df = (
    inspect_manual_review_rows(review_needed_df)
    if "review_needed_df" in globals()
    else pd.DataFrame()
)
display(manual_review_decisions_df)


#### 2.6.4 Finalize Current Chunk Decisions

After reviewing the current chunk, combine the model decisions with your interactive manual-review overrides for this chunk.


In [ ]:
if "current_chunk_results_df" in globals():
    accepted_chunk_decisions_df, reviewed_rows_df = finalize_review_decisions(
        current_chunk_results_df,
        manual_review_decisions_df if "manual_review_decisions_df" in globals() else None,
    )

    print("Rows manually reviewed in current chunk:", len(reviewed_rows_df))
    display(reviewed_rows_df)
else:
    print("No chunk results loaded yet.")


### 2.7 Save Semantic-Dedup Output

Save a second final dataset for the semantic-dedup stage, separate from the exact-dedup output.
Suggested name: `human_comments_semantic_dedup.parquet`.


In [ ]:
semantic_dedup_output_path = DATA_DIR / "human_comments_semantic_dedup.parquet"

semantic_dedup_comments_df.to_parquet(semantic_dedup_output_path, index=False)

print("Saved:")
print(semantic_dedup_output_path)